In [1]:
import torch
import requests

print("="*80)
print(f"{'PHASE 1: THE FUEL (DATA & TOKENIZATION)':^80}")
print("="*80)

# 1. DOWNLOAD THE DATA
print("[Step 1] Downloading 'Tiny Shakespeare'...")
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text = requests.get(url).text
print(f"✅ Success! Downloaded {len(text):,} characters of Shakespeare.")

# 2. BUILD THE VOCABULARY
# Find every unique character used in the text (letters, punctuation, newlines)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"\n[Step 2] Building Vocabulary...")
print(f"Total unique characters found (Vocab Size): {vocab_size}")
print(f"The Dictionary: '{''.join(chars)}'")

# 3. CREATE THE TRANSLATOR (ENCODER/DECODER)
# Create a mapping from character to integer (e.g., 'a' -> 39) and vice versa
stoi = {ch: i for i, ch in enumerate(chars)} # String to Integer
itos = {i: ch for i, ch in enumerate(chars)} # Integer to String

# The actual translation functions
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 4. TEST THE TRANSLATOR
print("\n[Step 3] Testing the Tokenizer...")
sample_text = "To be, or not to be"
encoded_math = encode(sample_text)
decoded_text = decode(encoded_math)

print(f"Original Text:  '{sample_text}'")
print(f"Encoded Tensors: {encoded_math}")
print(f"Decoded Back:   '{decoded_text}'")

# 5. CONVERT ENTIRE DATASET TO PYTORCH TENSOR
data = torch.tensor(encode(text), dtype=torch.long)
print(f"\n[Step 4] Final Dataset Ready!")
print(f"Shape of entire Shakespeare tensor: {list(data.shape)}")
print(f"First 20 tokens look like: {data[:20].tolist()}")
print("="*80)

                    PHASE 1: THE FUEL (DATA & TOKENIZATION)                     
[Step 1] Downloading 'Tiny Shakespeare'...
✅ Success! Downloaded 1,115,394 characters of Shakespeare.

[Step 2] Building Vocabulary...
Total unique characters found (Vocab Size): 65
The Dictionary: '
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'

[Step 3] Testing the Tokenizer...
Original Text:  'To be, or not to be'
Encoded Tensors: [32, 53, 1, 40, 43, 6, 1, 53, 56, 1, 52, 53, 58, 1, 58, 53, 1, 40, 43]
Decoded Back:   'To be, or not to be'

[Step 4] Final Dataset Ready!
Shape of entire Shakespeare tensor: [1115394]
First 20 tokens look like: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56]


In [2]:
import torch

# --- 1. DEVICE SETUP FOR RTX 3050 ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Make sure you see your RTX 3050 listed above!\n")

# --- 2. HYPERPARAMETERS (Tuned for RTX 3050 4GB/8GB) ---
batch_size = 32 # How many independent sequences to process in parallel
block_size = 64 # Maximum context length for predictions

# --- 3. TRAIN / VAL SPLIT ---
# We keep 10% of the data hidden from the model to test if it's actually learning
# or just blindly memorizing the text.
n = int(0.9 * len(data)) 
train_data = data[:n]
val_data = data[n:]

# --- 4. THE DATA LOADER ---
def get_batch(split):
    # Select the right dataset
    d = train_data if split == 'train' else val_data
    
    # Generate 32 random starting indices for our batch
    ix = torch.randint(len(d) - block_size, (batch_size,))
    
    # Stack the chunks into tensors (X is input, Y is the target shifted by 1)
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    
    # Shoot the data straight into your RTX 3050's VRAM!
    x, y = x.to(device), y.to(device)
    return x, y

# --- 5. TEST THE LOADER ---
xb, yb = get_batch('train')
print("[Data Loader Test]")
print(f"Inputs (X) shape:  {list(xb.shape)} -> (batch_size, block_size)")
print(f"Targets (Y) shape: {list(yb.shape)} -> (batch_size, block_size)")

print("\n--- Visualizing the Offset ---")
print("Notice how the Target (Y) is just the Context (X) shifted by 1 character:")
print(f"Context (X): '{decode(xb[0].tolist())}'")
print(f"Target  (Y): '{decode(yb[0].tolist())}'")

Using device: cuda
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
Make sure you see your RTX 3050 listed above!

[Data Loader Test]
Inputs (X) shape:  [32, 64] -> (batch_size, block_size)
Targets (Y) shape: [32, 64] -> (batch_size, block_size)

--- Visualizing the Offset ---
Notice how the Target (Y) is just the Context (X) shifted by 1 character:
Context (X): 'Hunting thee hence with hunt's-up to the day,
O, now be gone; mo'
Target  (Y): 'unting thee hence with hunt's-up to the day,
O, now be gone; mor'


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

print("="*80)
print(f"{'THE ULTIMATE MICRO-LLM TRAINING RUN (RTX 3050 EDITION)':^80}")
print("="*80)

# --- 1. HARDWARE & SCALED HYPERPARAMETERS ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32      # Number of sequences processed in parallel
block_size = 128     # Doubled context window!
n_embd = 384         # Scaled up embedding dimension
n_heads = 6          # 6 Query Heads
n_kv_heads = 2       # 2 KV Heads (GQA: 3 Q heads share 1 KV head)
n_layers = 6         # 6 Full Transformer Blocks
max_iters = 5000     # Pushing the GPU for a longer run
eval_interval = 500  # How often we evaluate and sample text
lr_muon = 0.02
lr_adam = 1e-3

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# --- 2. THE MODERN LLM ARCHITECTURE (LLAMA-3 CLONE) ---
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        return self.weight * (x * torch.rsqrt(norm_x + self.eps))

def precompute_rope_angles(dim, seq_len, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    positions = torch.arange(seq_len, device=device).float()
    angles = torch.outer(positions, inv_freq)
    freqs_cos = torch.cos(angles).repeat_interleave(2, dim=-1)
    freqs_sin = torch.sin(angles).repeat_interleave(2, dim=-1)
    return freqs_cos, freqs_sin

def apply_rope(x, cos, sin):
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    x_rotated = torch.stack([-x2, x1], dim=-1).flatten(-2)
    return (x * cos) + (x_rotated * sin)

class GQA_Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.head_dim = n_embd // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.wq = nn.Linear(n_embd, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(n_embd, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(n_embd, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, cos, sin, kv_cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, n_kv_heads, self.head_dim).transpose(1, 2)
        
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])
        
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_kv_cache = (k.detach(), v.detach())
        
        k_expanded = torch.repeat_interleave(k, repeats=self.n_rep, dim=1)
        v_expanded = torch.repeat_interleave(v, repeats=self.n_rep, dim=1)
        
        is_causal = kv_cache is None
        out = F.scaled_dot_product_attention(q, k_expanded, v_expanded, is_causal=is_causal)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.wo(out), new_kv_cache

class SwiGLU(nn.Module):
    def __init__(self):
        super().__init__()
        hidden_dim = 64 * ((int(4 * n_embd * (2/3)) + 64 - 1) // 64) 
        self.w_gate = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w_up   = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, n_embd, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class ModernBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = GQA_Attention()
        self.ffn = SwiGLU()
        self.norm1 = RMSNorm(n_embd)
        self.norm2 = RMSNorm(n_embd)
    def forward(self, x, cos, sin, kv_cache=None):
        attn_out, new_cache = self.attn(self.norm1(x), cos, sin, kv_cache)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, new_cache

class LlamaClone(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([ModernBlock() for _ in range(n_layers)])
        self.norm_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        # FIX: increased from block_size * 2 (256) to 4096 so that generation of
        # long sequences doesn't silently zero-out K keys and desync K/V lengths.
        self.cos, self.sin = precompute_rope_angles(n_embd // n_heads, 4096)

    def forward(self, idx, targets=None, past_caches=None):
        B, T = idx.shape
        x = self.token_emb(idx)
        offset = past_caches[0][0].shape[2] if past_caches is not None else 0
        cos, sin = self.cos[offset:offset+T], self.sin[offset:offset+T]
        
        new_caches = []
        for i, block in enumerate(self.blocks):
            layer_cache = past_caches[i] if past_caches is not None else None
            x, new_layer_cache = block(x, cos, sin, layer_cache)
            new_caches.append(new_layer_cache)
            
        x = self.norm_f(x)
        logits = self.lm_head(x)
        
        loss, acc = None, None
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
            # Calculate next-token accuracy
            preds = torch.argmax(logits_flat, dim=-1)
            acc = (preds == targets_flat).float().mean()
            
        return logits, loss, acc, new_caches

    @torch.no_grad()
    def generate_fast(self, idx, max_new_tokens):
        self.eval()
        logits, _, _, caches = self(idx)
        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        idx = torch.cat((idx, next_token), dim=1)
        
        for _ in range(max_new_tokens - 1):
            logits, _, _, caches = self(next_token, past_caches=caches)
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            idx = torch.cat((idx, next_token), dim=1)
            
        self.train()
        return idx

# --- 3. MODEL, OPTIMIZER, AND ROUTING ---
model = LlamaClone().to(device)
print(f"✅ Model Assembled! Total params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(model)

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95):
        super().__init__(params, dict(lr=lr, momentum=momentum))
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                G = p.grad
                state = self.state[p]
                if 'buf' not in state: state['buf'] = torch.zeros_like(G)
                buf = state['buf']
                buf.mul_(group['momentum']).add_(G, alpha=1 - group['momentum'])
                X_opt = buf
                m, n = X_opt.shape
                transposed = m > n
                if transposed: X_opt = X_opt.T
                X_opt = X_opt / (torch.linalg.matrix_norm(X_opt) + 1e-8)
                for _ in range(5):
                    A = X_opt @ X_opt.T
                    X_opt = 1.5 * X_opt - 0.5 * (A @ X_opt)
                if transposed: X_opt = X_opt.T
                p.add_(X_opt, alpha=-group['lr'])

muon_params, adamw_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    if len(p.shape) == 2 and 'emb' not in name and 'head' not in name:
        muon_params.append(p)
    else:
        adamw_params.append(p)

opt_muon = Muon(muon_params, lr=lr_muon)
opt_adamw = torch.optim.AdamW(adamw_params, lr=lr_adam)

# --- 4. THE BEFORE SNAPSHOT ---
print("\n" + "="*80)
print("UNTRAINED MODEL GENERATION (Absolute Gibberish):")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate_fast(context, max_new_tokens=150)[0].tolist()))
print("="*80 + "\n")

# --- 5. THE TRAINING LOOP ---
tracked_iters, tracked_loss, tracked_acc = [], [], []

@torch.no_grad()
def evaluate():
    model.eval()
    out_loss, out_acc = 0.0, 0.0
    for _ in range(50): # Average over 50 batches
        X, Y = get_batch('val')
        _, loss, acc, _ = model(X, targets=Y)
        out_loss += loss.item()
        out_acc += acc.item()
    model.train()
    return out_loss / 50, out_acc / 50

print(f"🔥 Firing up the RTX 3050 for {max_iters} steps...")
start_time = time.time()

for iter in range(max_iters):
    # Evaluation and Mid-Training Inference
    if iter % eval_interval == 0 or iter == max_iters - 1:
        val_loss, val_acc = evaluate()
        tracked_iters.append(iter)
        tracked_loss.append(val_loss)
        tracked_acc.append(val_acc * 100) # Convert to percentage
        
        print(f"Step {iter:4d} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc*100:.2f}%")
        
        # Mid-training generation snapshot (Only every 1000 steps so we don't spam the console)
        if iter > 0 and iter % 1000 == 0:
            print("-" * 50)
            print(f"Mid-Training Snapshot (Step {iter}):")
            sample = decode(model.generate_fast(context, max_new_tokens=100)[0].tolist())
            print(sample.replace('\n', ' ')) # Print on one line for neatness
            print("-" * 50)

    # Forward Pass
    xb, yb = get_batch('train')
    logits, loss, acc, _ = model(xb, targets=yb)
    
    # Backward Pass & Optimize
    opt_muon.zero_grad(set_to_none=True)
    opt_adamw.zero_grad(set_to_none=True)
    loss.backward()
    opt_muon.step()
    opt_adamw.step()

print(f"\n✅ Training Complete in {(time.time() - start_time):.2f} seconds.")

# --- 6. THE AFTER SNAPSHOT ---
print("\n" + "="*80)
print("FULLY TRAINED MODEL GENERATION:")
# Give it a prompt this time!
prompt = "ROMEO:\nWhere art thou"
encoded_prompt = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
final_text = decode(model.generate_fast(encoded_prompt, max_new_tokens=300)[0].tolist())
print(final_text)
print("="*80 + "\n")

# --- 7. PLOTTING LOSS AND ACCURACY ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss Plot
ax1.plot(tracked_iters, tracked_loss, color='#e74c3c', linewidth=2, marker='o')
ax1.set_title("Validation Loss (Lower is Better)", fontweight='bold')
ax1.set_xlabel("Training Steps")
ax1.set_ylabel("Cross Entropy Loss")
ax1.grid(True, alpha=0.3)

# Accuracy Plot
ax2.plot(tracked_iters, tracked_acc, color='#2ecc71', linewidth=2, marker='o')
ax2.set_title("Next-Token Accuracy (Higher is Better)", fontweight='bold')
ax2.set_xlabel("Training Steps")
ax2.set_ylabel("Accuracy (%)")
ax2.grid(True, alpha=0.3)

plt.suptitle("Micro Llama-3 Clone Training Metrics", fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()


             THE ULTIMATE MICRO-LLM TRAINING RUN (RTX 3050 EDITION)             
✅ Model Assembled! Total params: 9.49M
LlamaClone(
  (token_emb): Embedding(65, 384)
  (blocks): ModuleList(
    (0-5): 6 x ModernBlock(
      (attn): GQA_Attention(
        (wq): Linear(in_features=384, out_features=384, bias=False)
        (wk): Linear(in_features=384, out_features=128, bias=False)
        (wv): Linear(in_features=384, out_features=128, bias=False)
        (wo): Linear(in_features=384, out_features=384, bias=False)
      )
      (ffn): SwiGLU(
        (w_gate): Linear(in_features=384, out_features=1024, bias=False)
        (w_up): Linear(in_features=384, out_features=1024, bias=False)
        (w_down): Linear(in_features=1024, out_features=384, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (norm_f): RMSNorm()
  (lm_head): Linear(in_features=384, out_features=65, bias=False)
)

UNTRAINED MODEL GENERATION (Absolute Gibberish):

pDev?Johsavf3DTyN.S:DTyN.S

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [6, 256] but got: [6, 257].